# AutoResearch

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))  # ensure autoresearch/ is importable

from autoresearch.config import BASELINE_F1, DATASET_SIZE
from autoresearch.data import load_split
from autoresearch.runner import generate_candidates, run_sweep
from autoresearch.leaderboard import save_result, clear_results, display_leaderboard, get_best_run
from autoresearch.error_analysis import export_hard_examples
from autoresearch.features import clear_embed_cache

print(f"Baseline to beat: {BASELINE_F1}")
print(f"Dataset size: {DATASET_SIZE}")

/home/akprajwal/ml/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Baseline to beat: 0.9
Dataset size: 10000


## Load data (deterministic split)

In [2]:
split = load_split()
print("Splits:", split.sizes)
import numpy as np
print(f"Positive rate — train: {split.y_train.mean():.3f}  "
      f"val: {split.y_val.mean():.3f}  test: {split.y_test.mean():.3f}")

Splits: train=6999, val=1500, test=1501
Positive rate — train: 0.496  val: 0.497  test: 0.496


## Run small sweep


In [3]:

clear_results()
clear_embed_cache()

SEARCH_SIZE = 'small'   # change to 'medium' for full

candidates = generate_candidates(search_size=SEARCH_SIZE)
print(f"Running {len(candidates)} candidates ({SEARCH_SIZE} search)...\n")

results = run_sweep(
    split=split,
    candidates=candidates,
    on_result=save_result,   # Save each results
    verbose=True,
)
print(f"\nSweep complete. {len(results)} runs finished.")

Cleared /home/akprajwal/ml/autoresearch_results/runs.jsonl
Running 4 candidates (small search)...


[1/4] run_id=s1  embed=all-MiniLM-L6-v2  word_ng=(1, 2)  char_ng=(3, 5)  C=4.0  fusion_c=1.0
  [embed cache hit] sentence-transformers_all-MiniLM-L6-v2_6999_f8ed8258f0f66c4e.npy
  [embed cache hit] sentence-transformers_all-MiniLM-L6-v2_1500_0f57cf01eafdca5d.npy
  [embed cache hit] sentence-transformers_all-MiniLM-L6-v2_1501_11427485dff49d1c.npy
    status=ok  val_f1=0.907395  thr=0.5247  tpr@0.1fpr=0.914094  time=279.3s

[2/4] run_id=s2  embed=bge-small-en-v1.5  word_ng=(1, 2)  char_ng=(3, 5)  C=4.0  fusion_c=1.0
  [embed cache hit] BAAI_bge-small-en-v1.5_6999_253b7daea4a91eed.npy
  [embed cache hit] BAAI_bge-small-en-v1.5_1500_99a5f5f613716e2d.npy
  [embed cache hit] BAAI_bge-small-en-v1.5_1501_6fdf1984cc72ab6c.npy
    status=ok  val_f1=0.913907  thr=0.5148  tpr@0.1fpr=0.926174  time=228.0s

[3/4] run_id=s3  embed=all-MiniLM-L6-v2  word_ng=(1, 3)  char_ng=(3, 6)  C=8.0  fusion_c=2.0
  

In [4]:
display_leaderboard(top_n=10)

=== AutoResearch Leaderboard (top 4) ===
  run_id                             embed_model word_ngram char_ngram  logreg_c  fusion_c  val_f1  val_accuracy  val_roc_auc  val_tpr_at_fpr_0.1000  val_tpr_at_fpr_0.0100  selected_threshold  beats_baseline  elapsed_s
0     s4                  BAAI/bge-small-en-v1.5     [1, 3]     [3, 6]    8.0000    2.0000  0.9239        0.9227       0.9698                 0.9450                 0.5477              0.4604            True   211.8000
1     s3  sentence-transformers/all-MiniLM-L6-v2     [1, 3]     [3, 6]    8.0000    2.0000  0.9196        0.9187       0.9711                 0.9369                 0.6134              0.4802            True   182.9000
2     s2                  BAAI/bge-small-en-v1.5     [1, 2]     [3, 5]    4.0000    1.0000  0.9139        0.9133       0.9656                 0.9262                 0.4966              0.5148            True   228.0000
3     s1  sentence-transformers/all-MiniLM-L6-v2     [1, 2]     [3, 5]    4.0000   

## Compare best AutoResearch run vs baseline

In [5]:
import pandas as pd
from autoresearch.leaderboard import get_leaderboard

best = get_best_run()
if best is None:
    print("No completed runs found.")
else:
    comparison_rows = [
        {
            "Model": "Baseline (TF_IDF.ipynb fused)",
            "val_f1": BASELINE_F1,
            "test_f1": "~0.91",
            "threshold": "calibrated",
            "notes": "Manual config, single run",
        },
        {
            "Model": f"AutoResearch best ({best['run_id']})",
            "val_f1": best.get("val_f1", "n/a"),
            "test_f1": best.get("test_f1", "(below baseline)"),
            "threshold": best.get("selected_threshold", "n/a"),
            "notes": (
                f"embed={best.get('embed_model','').split('/')[-1]}  "
                f"word_ng={best.get('word_ngram')}  "
                f"char_ng={best.get('char_ngram')}  "
                f"C={best.get('logreg_c')}  fusion_c={best.get('fusion_c')}"
            ),
        },
    ]
    print("\n=== Baseline vs AutoResearch Best ===")
    display(pd.DataFrame(comparison_rows))

    print("\nBest run full val metrics:")
    for k, v in sorted(best.items()):
        if k.startswith("val_") or k in ("selected_threshold", "beats_baseline"):
            print(f"  {k:45s}: {v}")


=== Baseline vs AutoResearch Best ===


,Model,val_f1,test_f1,threshold,notes
0,Baseline (TF_IDF.ipynb fused),0.9000,~0.91,calibrated,"Manual config, single run"
1,AutoResearch best (s4),0.9239,0.9243,0.4604,"embed=bge-small-en-v1.5 word_ng=[1, 3] char_..."



Best run full val metrics:
  beats_baseline                               : True
  selected_threshold                           : 0.4604
  val_accuracy                                 : 0.922667
  val_f1                                       : 0.923885
  val_f1_calibrated                            : 0.923885
  val_pr_auc                                   : 0.968895
  val_precision                                : 0.903723
  val_recall                                   : 0.944966
  val_roc_auc                                  : 0.969838
  val_tfidf_accuracy                           : 0.918
  val_tfidf_f1                                 : 0.918812
  val_tfidf_pr_auc                             : 0.969675
  val_tfidf_precision                          : 0.903896
  val_tfidf_recall                             : 0.934228
  val_tfidf_roc_auc                            : 0.970679
  val_tfidf_threshold                          : 0.5445
  val_tfidf_tpr_at_fpr_0.0010                  : 0.3906

## Error mining on best run

Retrain best config and export hard false negatives/positives for targeted data augmentation review.

In [6]:
from autoresearch.config import RunConfig
from autoresearch.runner import run_single
from autoresearch.features import TfidfBranch, EmbedXGBBranch, FusionModel
from autoresearch.evaluate import select_threshold
import numpy as np

if best is None:
    print("No best run available.")
else:
    # Retrain best config to get probability arrays for error mining
    bcfg = RunConfig(
        run_id=best['run_id'],
        word_ngram=tuple(best['word_ngram']),
        char_ngram=tuple(best['char_ngram']),
        logreg_c=best['logreg_c'],
        embed_model=best['embed_model'],
        xgb={
            'n_estimators': best.get('xgb_n_estimators', 350),
            'max_depth': best.get('xgb_max_depth', 7),
            'learning_rate': best.get('xgb_learning_rate', 0.05),
            'subsample': best.get('xgb_subsample', 0.90),
            'colsample_bytree': best.get('xgb_colsample_bytree', 0.90),
            'reg_lambda': best.get('xgb_reg_lambda', 1.0),
        },
        fusion_c=best['fusion_c'],
    )

    tfidf = TfidfBranch(word_ngram=bcfg.word_ngram, char_ngram=bcfg.char_ngram, logreg_c=bcfg.logreg_c)
    tfidf.fit_transform(split.x_train, split.y_train)
    val_prob_tfidf  = tfidf.predict_proba(split.x_val)
    test_prob_tfidf = tfidf.predict_proba(split.x_test)

    xgb_b = EmbedXGBBranch(model_id=bcfg.embed_model, xgb_params=bcfg.xgb)
    xgb_b.fit(split.x_train, split.y_train)
    val_prob_xgb  = xgb_b.predict_proba(split.x_val)
    test_prob_xgb = xgb_b.predict_proba(split.x_test)

    fusion = FusionModel(c=bcfg.fusion_c)
    fusion.fit(val_prob_tfidf, val_prob_xgb, split.y_val)
    test_prob_fusion = fusion.predict_proba(test_prob_tfidf, test_prob_xgb)

    thr = best.get('selected_threshold', 0.5)

    export_hard_examples(
        run_id=bcfg.run_id,
        y_true=split.y_test,
        y_prob=test_prob_fusion,
        rows=split.rows_test,
        threshold=thr,
    )
    print(f"Hard examples exported. Threshold used: {thr}")

Saved hard examples → /home/akprajwal/ml/autoresearch_results/errors_s4.json
Hard examples exported. Threshold used: 0.4604
